# Audiovisual Speech Recognition Prototype

This notebook sketches a character-level audiovisual recognition pipeline: log-Mel audio features and grayscale video clips are encoded separately, fused with a Transformer, and trained with CTC loss.

**Scope:** the included dataset is synthetic (sine waves/noise, moving squares, and random character strings). It only exercises tensor shapes and the training loop. It does not contain speech, real lip movements, a held-out evaluation set, or evidence of recognition accuracy.

Install dependencies from `requirements.txt` first. For GPU use, install a matching PyTorch and TorchAudio build using the [official PyTorch selector](https://pytorch.org/get-started/locally/). The demo is opt-in: set `RUN_DEMO = True` in the final cell to run it.


In [ ]:
# Prototype: audio/video feature fusion with a CTC training loop.
# See README.md for dependency setup and limitations.

# -----------------------
# Imports & utilities
# -----------------------
import math
import random
from dataclasses import dataclass

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchaudio.transforms import MelSpectrogram, AmplitudeToDB

# -----------------------
# Config
# -----------------------
@dataclass
class CFG:
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42

    # Audio
    sample_rate: int = 16000
    n_mels: int = 80
    win_length: int = 400      # 25ms @16k
    hop_length: int = 160      # 10ms @16k
    audio_seconds: float = 2.0  # 2 sec audio -> spectrogram time frames ~200

    # Video (lips)
    n_frames: int = 16        # number of frames in clip
    frame_h: int = 64
    frame_w: int = 64

    # Model dims
    audio_feat_dim: int = 256
    video_feat_dim: int = 256
    fusion_dim: int = 512
    transformer_nhead: int = 8
    transformer_layers: int = 3

    # Training
    vocab: str = " abcdefghijklmnopqrstuvwxyz'"  # char-level vocabulary (space + apostrophe)
    blank_id: int = 0
    batch_size: int = 8
    lr: float = 3e-4
    epochs: int = 3  # small for demo
    max_target_len: int = 16  # max chars per transcript (dummy)
    clip_grad_norm: float = 1.0


cfg = CFG()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)

# Build vocab mapping (char -> int). Reserve 0 for CTC blank
vocab_chars = list(cfg.vocab)
stoi = {c:i+1 for i,c in enumerate(vocab_chars)}   # start from 1
itos = {i+1:c for i,c in enumerate(vocab_chars)}
blank_id = 0
vocab_size = len(stoi) + 1  # +1 for blank

# -----------------------
# Synthetic dataset (dummy)
# -----------------------
class DummyAudioVisualDataset(Dataset):
    """
    Creates synthetic examples:
      - audio: random noise shaped like real spectrogram
      - video: random moving square as lip-region
      - transcript: random short word from vocab
    Purpose: test the pipeline end-to-end.
    """
    def __init__(self, n_samples=200):
        self.n = n_samples
        self.audio_len = int(cfg.sample_rate * cfg.audio_seconds)
        self.melspec = MelSpectrogram(
            sample_rate=cfg.sample_rate,
            n_fft=cfg.win_length,
            win_length=cfg.win_length,
            hop_length=cfg.hop_length,
            n_mels=cfg.n_mels
        )
        self.to_db = AmplitudeToDB()
        # Pre-generate
        self.data = [self._create_sample(i) for i in range(self.n)]

    def _create_sample(self, idx):
        # --- audio: synthetic sine + noise
        t = np.linspace(0, cfg.audio_seconds, self.audio_len, endpoint=False)
        freq = np.random.uniform(100, 800)
        audio = 0.1*np.sin(2*np.pi*freq*t).astype(np.float32)
        audio += 0.02*np.random.randn(self.audio_len).astype(np.float32)
        audio_tensor = torch.tensor(audio)

        # compute log-mel (will be [n_mels, time_frames])
        m = self.melspec(audio_tensor)
        m = self.to_db(m)  # log-mel

        # --- video: moving square in grayscale frames
        frames = np.zeros((cfg.n_frames, cfg.frame_h, cfg.frame_w), dtype=np.float32)
        # moving square:
        sx, sy = np.random.randint(4, 12), np.random.randint(4, 12)
        for f in range(cfg.n_frames):
            cx = int((cfg.frame_w-1) * ((f / cfg.n_frames) * 0.8 + 0.1)) + np.random.randint(-2,3)
            cy = int((cfg.frame_h-1) * (0.5 + 0.1*np.sin(2*math.pi*f/cfg.n_frames)))
            x0, x1 = max(0,cx-sx), min(cfg.frame_w, cx+sx)
            y0, y1 = max(0,cy-sy), min(cfg.frame_h, cy+sy)
            frames[f,y0:y1,x0:x1] = 0.8 + 0.2*np.random.rand()
            # add background noise
            frames[f] += 0.05 * np.random.randn(cfg.frame_h, cfg.frame_w)
        frames = np.clip(frames, 0.0, 1.0)
        video_tensor = torch.tensor(frames)  # [T,H,W]

        # --- transcript: random short sequence of chars
        tgt_len = np.random.randint(3, cfg.max_target_len+1)
        chars = [random.choice(vocab_chars) for _ in range(tgt_len)]
        transcript = "".join(chars)
        # convert to indices excluding blank (CTC blank id=0)
        tgt_ids = [stoi[ch] for ch in transcript]

        return {
            "audio": m,            # [n_mels, time]
            "video": video_tensor, # [T, H, W]
            "transcript": transcript,
            "tgt_ids": torch.tensor(tgt_ids, dtype=torch.long)
        }

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return self.data[idx]

# collate function to pad spectrogram/time lengths
def collate_fn(batch):
    # audio specs have shape [n_mels, T_i]
    max_T = max(x["audio"].shape[1] for x in batch)
    max_vidT = max(x["video"].shape[0] for x in batch)
    audios = []
    videos = []
    targets = []
    target_lens = []
    input_lens = []

    for x in batch:
        a = x["audio"]
        # pad time axis (right)
        pad = max_T - a.shape[1]
        if pad > 0:
            a = F.pad(a, (0,pad))
        audios.append(a.unsqueeze(0))  # 1 x n_mels x T
        # video: pad frames if needed
        v = x["video"]
        padv = max_vidT - v.shape[0]
        if padv > 0:
            v = F.pad(v, (0,0,0,0,0,padv))  # pad first dim (frames) at end
        videos.append(v.unsqueeze(0))  # 1 x T x H x W
        targets.append(x["tgt_ids"])
        target_lens.append(len(x["tgt_ids"]))
        # input length for CTC = time frames after encoder; we compute later using model
        input_lens.append(a.shape[1])

    audios = torch.cat(audios, dim=0)     # B x n_mels x T
    videos = torch.cat(videos, dim=0)     # B x T x H x W
    targets_cat = torch.cat(targets, dim=0)  # sum(target_lens)
    return {
        "audio": audios,
        "video": videos,
        "targets": targets_cat,
        "target_lens": torch.tensor(target_lens, dtype=torch.long),
        "input_frame_lens": torch.tensor(input_lens, dtype=torch.long)  # raw spectrogram frames per example
    }

# -----------------------
# Model components
# -----------------------
class AudioEncoder(nn.Module):
    """
    Simple 2D CNN encoder over log-mel spectrogram. Produces per-time-step features.
    Input: B x n_mels x T
    Output: B x T' x feat (we keep time dimension as last dimension)
    """
    def __init__(self, n_mels=cfg.n_mels, out_dim=cfg.audio_feat_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(3,3), padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=(3,3), stride=(2,1), padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=(3,3), stride=(2,1), padding=1), nn.BatchNorm2d(128), nn.ReLU(),
        )
        # after conv, collapse mel-dim -> features, keep time dimension (width)
        self.project = nn.Conv1d(128 * ((n_mels + 3) // 4), out_dim, kernel_size=1)

    def forward(self, x):
        # x: B x n_mels x T  --> add channel dim
        B, M, T = x.shape
        x = x.unsqueeze(1)  # B x 1 x M x T
        x = self.conv(x)    # B x C x M' x T'
        C, Mp, Tp = x.shape[1], x.shape[2], x.shape[3]
        # merge channel + mel axis into feature axis, keep time axis (Tp)
        x = x.permute(0,3,1,2).contiguous()   # B x Tp x C x Mp
        x = x.view(B, Tp, C*Mp)               # B x Tp x (C*Mp)
        # project to desired dim
        x = self.project(x.permute(0,2,1))    # B x out_dim x Tp
        x = x.permute(0,2,1)                  # B x Tp x out_dim
        return x  # per-frame features (time-major preserved)

class VideoEncoder(nn.Module):
    """
    Small 3D CNN to encode short lip clips.
    Input: B x T x H x W  (grayscale)
    Output: B x T' x feat  (we will spatially pool keeping time dim roughly aligned)
    """
    def __init__(self, in_ch=1, out_dim=cfg.video_feat_dim):
        super().__init__()
        self.net3d = nn.Sequential(
            nn.Conv3d(in_ch, 16, kernel_size=(3,5,5), padding=(1,2,2), stride=(1,2,2)), nn.ReLU(),
            nn.Conv3d(16, 32, kernel_size=(3,3,3), padding=1, stride=(1,2,2)), nn.ReLU(),
            nn.Conv3d(32, 64, kernel_size=(3,3,3), padding=1, stride=(1,1,1)), nn.ReLU(),
        )
        # final projection
        self.pool = nn.AdaptiveAvgPool3d((None,1,1))  # preserve temporal dim, spatial -> 1x1
        self.project = nn.Linear(64, out_dim)

    def forward(self, x):
        # x: B x T x H x W -> convert to B x C x T x H x W
        B, T, H, W = x.shape
        x = x.unsqueeze(1)  # B x 1 x T x H x W
        x = self.net3d(x)   # B x C x T' x H' x W'
        # pool spatially to 1x1, keep time dim
        x = self.pool(x)    # B x C x T' x 1 x 1
        x = x.squeeze(-1).squeeze(-1)  # B x C x T'
        x = x.permute(0,2,1)  # B x T' x C
        x = self.project(x)   # B x T' x out_dim
        return x

class FusionASRModel(nn.Module):
    """
    Combine audio & video encoders, fuse features, run transformer, predict char logits per frame for CTC.
    """
    def __init__(self, vocab_size, blank_id=0):
        super().__init__()
        self.audio_enc = AudioEncoder(n_mels=cfg.n_mels, out_dim=cfg.audio_feat_dim)
        self.video_enc = VideoEncoder(in_ch=1, out_dim=cfg.video_feat_dim)
        self.fusion_proj = nn.Linear(cfg.audio_feat_dim + cfg.video_feat_dim, cfg.fusion_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=cfg.fusion_dim, nhead=cfg.transformer_nhead, dim_feedforward=cfg.fusion_dim*4, dropout=0.1, activation="relu")
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.transformer_layers)
        self.classifier = nn.Linear(cfg.fusion_dim, vocab_size)  # includes blank
        self.blank_id = blank_id

    def forward(self, audio_spec, video_frames):
        """
        audio_spec: B x n_mels x T_spec
        video_frames: B x T_vid x H x W
        Returns: logits: T_out x B x vocab_size (required by CTC in PyTorch: T x B x C)
                 input_lengths: length per example (time steps)
        """
        # audio features
        a_feats = self.audio_enc(audio_spec.to(audio_spec.device))  # B x Ta x audio_feat_dim
        # video features
        v_feats = self.video_enc(video_frames.to(video_frames.device))  # B x Tv x video_feat_dim

        # If time dims differ, we align by up/downsampling the shorter one to match the longer
        Ta = a_feats.shape[1]
        Tv = v_feats.shape[1]
        if Ta != Tv:
            # resample via linear interpolation on time axis
            if Ta > Tv:
                # upsample v to Ta
                v_feats = F.interpolate(v_feats.permute(0,2,1), size=Ta, mode="linear", align_corners=False).permute(0,2,1)
            else:
                # upsample a to Tv
                a_feats = F.interpolate(a_feats.permute(0,2,1), size=Tv, mode="linear", align_corners=False).permute(0,2,1)

        # now both have same time length
        T = a_feats.shape[1]
        fused = torch.cat([a_feats, v_feats], dim=-1)  # B x T x (A+V)
        fused = self.fusion_proj(fused)                # B x T x fusion_dim

        # transformer expects (T, B, D)
        x = fused.permute(1,0,2)
        x = self.transformer(x)   # T x B x D
        logits = self.classifier(x)  # T x B x vocab
        # return logits (T,B,C)
        return logits, torch.full((audio_spec.shape[0],), T, dtype=torch.long, device=audio_spec.device)

# -----------------------
# Training utilities
# -----------------------
def greedy_ctc_decode(logits):
    # logits: T x B x C (torch)
    probs = torch.argmax(logits, dim=-1)  # T x B
    preds = []
    for b in range(probs.shape[1]):
        seq = probs[:,b].cpu().numpy().tolist()
        # collapse repeats and remove blanks(0)
        out = []
        prev = None
        for s in seq:
            if s != prev and s != blank_id:
                out.append(itos.get(int(s), "?"))
            prev = s
        preds.append("".join(out))
    return preds

# -----------------------
# End-to-end train loop (demo)
# -----------------------
def train_demo():
    device = cfg.device
    print("Device:", device)
    ds = DummyAudioVisualDataset(n_samples=256)
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)

    model = FusionASRModel(vocab_size=vocab_size, blank_id=blank_id).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    ctc_loss = nn.CTCLoss(blank=blank_id, zero_infinity=True)

    model.train()
    for epoch in range(1, cfg.epochs+1):
        epoch_loss = 0.0
        for batch_idx, batch in enumerate(loader):
            audio = batch["audio"].to(device)          # B x n_mels x T_spec
            video = batch["video"].to(device)          # B x T_vid x H x W
            targets = batch["targets"].to(device)      # sumL
            target_lens = batch["target_lens"].to(device)  # B
            # forward
            logits, input_lens = model(audio, video)   # logits: T x B x C
            T, B, C = logits.shape
            log_probs = F.log_softmax(logits, dim=-1)  # T x B x C
            # prepare for CTC: flatten targets are already concatenated in collate_fn
            # input_lens is uniform in this implementation (T) per example
            input_lengths = input_lens.to(device)      # B  (time per example)
            # But CTC in PyTorch expects input_lengths in number of time steps (T) per example.
            # Our model returns same length T for all examples (we gave full length), so set to T for each.
            input_lengths = torch.full((B,), T, dtype=torch.long, device=device)

            # targets must be 1D concatenated + target_lengths
            # we already have targets concatenated and target_lens
            loss = ctc_loss(log_probs, targets, input_lengths, target_lens)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad_norm)
            optimizer.step()
            epoch_loss += loss.item()

            if (batch_idx+1) % 10 == 0:
                # quick decode a sample
                model.eval()
                with torch.no_grad():
                    sample_logits, _ = model(audio[:2], video[:2])
                    preds = greedy_ctc_decode(sample_logits)
                    refs = []
                    # reconstruct reference strings from targets slice
                    idx = 0
                    for l in target_lens[:2].tolist():
                        refs.append("".join([itos[int(x.item())] for x in targets[idx:idx+l]]))
                        idx += l
                print(f"Epoch {epoch} Batch {batch_idx+1} Loss {loss.item():.4f} Ref0='{refs[0]}' Pred0='{preds[0]}'")
                model.train()

        print(f"Epoch {epoch} avg loss: {epoch_loss/len(loader):.4f}")

    # final demo: run on 4 samples
    model.eval()
    with torch.no_grad():
        for i in range(4):
            ex = ds[i]
            a = ex["audio"].unsqueeze(0).to(device)
            v = ex["video"].unsqueeze(0).to(device)
            logits, _ = model(a, v)
            pred = greedy_ctc_decode(logits)[0]
            print(f"Sample {i} ref='{ex['transcript']}' pred='{pred}'")

# Run explicitly after reviewing the synthetic-data limitations.
RUN_DEMO = False
if RUN_DEMO:
    train_demo()
